In [ ]:
!pip install pandas matplotlib seaborn datasets pyarrow

In [ ]:
import pandas as pd
import json
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_dataset
from collections import Counter

# DATA PREPARATION
def get_german_dataset():
    ds = load_dataset("gretelai/synthetic_pii_finance_multilingual", split="train")
    df = ds.to_pandas()
    df_de = df[df['language'] == 'German'].copy()
    return df_de.dropna(subset=['generated_text'])

# ANALYSIS PIPELINE
def run_eda():
    df = get_german_dataset()
    
    # Text quality metrics
    df['word_count'] = df['generated_text'].str.split().str.len()
    df['pii_count'] = df['pii_spans'].apply(lambda x: len(json.loads(x)) if isinstance(x, str) else len(x))
    
    print(f"--- Data Summary ---")
    print(f"Total Documents: {len(df)}")
    print(f"Avg words per doc: {df['word_count'].mean():.2f}")
    print(f"Low quality entries: {len(df[df['quality_score'] < 0.5])}")
    print(f"Duplicates found: {df.duplicated(subset=['generated_text']).sum()}")

    # PII Category analysis
    all_labels = []
    for spans in df['pii_spans']:
        data = json.loads(spans) if isinstance(spans, str) else spans
        all_labels.extend([s['label'] for s in data])
    
    label_df = pd.DataFrame.from_dict(Counter(all_labels), orient='index', columns=['count'])
    label_df = label_df.sort_values('count', ascending=False)

    # Visualization
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    # Word count distribution
    sns.histplot(df['word_count'], ax=axes[0], color='skyblue')
    axes[0].set_title('Document Length (Words)')
    
    # PII count per document
    sns.boxplot(y=df['pii_count'], ax=axes[1], color='orange')
    axes[1].set_title('PII Density per Doc')
    
    # PII category distribution
    sns.barplot(x=label_df.index, y=label_df['count'], ax=axes[2], palette='viridis')
    axes[2].set_title('PII Category Frequency')
    axes[2].tick_params(axis='x', rotation=45)
    
    plt.tight_layout()
    plt.show()

if __name__ == "__main__":
    run_eda()